# Data Preparation

This notebook prepares **NF-UNSW-NB15-v2** and **NF-CSE-CIC-IDS2018-v2** for the intrusion detection experiments. It checks the shared columns, loads manageable samples, and prepares **43 model features**.

**To run:** Open in Google Colab, connect Google Drive, and run the cells in order. Dataset files are expected in `MyDrive/datanew/`; existing samples in `MyDrive/datanew/a4_analysis/` are reused.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Running outside Colab: edit DATA_ROOT and ANALYSIS_ROOT below.')

# Dataset locations in Google Drive.
DATA_ROOT = Path('/content/drive/MyDrive/datanew')
ANALYSIS_ROOT = DATA_ROOT / 'a4_analysis'
ANALYSIS_ROOT.mkdir(parents=True, exist_ok=True)

RAW_FILES = {
    'UNSW': DATA_ROOT / 'NF-UNSW-NB15-v2.csv',
    'CIC': DATA_ROOT / 'NF-CSE-CIC-IDS2018-v2.csv',
}
SAMPLE_FILES = {
    'UNSW': ANALYSIS_ROOT / 'UNSW_100k.csv',
    'CIC': ANALYSIS_ROOT / 'CIC_100k.csv',
}
# Dataset sizes for sampling if saved files are missing.
RAW_ROW_COUNTS = {'UNSW': 2_390_275, 'CIC': 18_893_708}
RANDOM_SEED = 786
TARGET_SAMPLE_SIZE = 100_000

for name in ('UNSW', 'CIC'):
    print(f'{name}: cached sample={SAMPLE_FILES[name].exists()}, '
          f'raw CSV={RAW_FILES[name].exists()}')

## Check dataset columns

Use the same feature names for both datasets. IP addresses and labels are not model inputs.


In [ ]:
EXCLUDED_COLUMNS = {'IPV4_SRC_ADDR', 'IPV4_DST_ADDR', 'Label', 'Attack'}
REQUIRED_LABEL_COLUMNS = {'Label', 'Attack'}


def available_dataset_path(name):
    if SAMPLE_FILES[name].is_file():
        return SAMPLE_FILES[name]
    if RAW_FILES[name].is_file():
        return RAW_FILES[name]
    raise FileNotFoundError(
        f'No {name} sample or raw CSV found. Expected one of: '
        f'{SAMPLE_FILES[name]} or {RAW_FILES[name]}'
    )

headers = {
    name: pd.read_csv(available_dataset_path(name), nrows=0).columns.tolist()
    for name in ('UNSW', 'CIC')
}
feature_cols = [col for col in headers['UNSW'] if col not in EXCLUDED_COLUMNS]

if len(feature_cols) != 41:
    raise ValueError(f'Expected 41 numeric features, got {len(feature_cols)}.')

for name, columns in headers.items():
    missing = (set(feature_cols) | REQUIRED_LABEL_COLUMNS) - set(columns)
    if missing:
        raise ValueError(f'{name} is missing expected columns: {sorted(missing)}')

print('Raw numerical features:', len(feature_cols))
print('Source and target schemas: verified')
print('IP addresses excluded from model features')

## Load dataset samples

The CIC file has millions of rows, so reading it all at once can use too much memory. We read the CSV files in chunks of 100,000 rows and randomly keep a small share from each chunk. This gives us roughly 100,000 records per dataset without loading the full files into memory.

The random seeds make sampling repeatable. If a saved sample already exists, we use it instead. These samples are not class-balanced.

In [ ]:
def build_or_load_sample(name, seed):
    cached = SAMPLE_FILES[name]
    if cached.is_file():
        print(f'{name}: loading existing sample: {cached.name}')
        return pd.read_csv(cached)

    raw = RAW_FILES[name]
    if not raw.is_file():
        raise FileNotFoundError(f'{name}: missing both cached sample and raw CSV')

    print(f'{name}: streaming raw CSV to create a ~100k-row sample')
    probability = TARGET_SAMPLE_SIZE / RAW_ROW_COUNTS[name]
    if not 0 < probability <= 1:
        raise ValueError('Invalid sampling probability')

    rng = np.random.default_rng(seed)
    chosen_chunks = []
    usecols = feature_cols + ['Label', 'Attack']

    for chunk in pd.read_csv(
        raw, usecols=usecols,
        dtype={col: 'float64' for col in feature_cols},
        chunksize=100_000,
    ):
        keep = rng.random(len(chunk)) < probability
        if keep.any():
            chosen_chunks.append(chunk.loc[keep].copy())

    if not chosen_chunks:
        raise RuntimeError(f'{name}: no records sampled')

    result = pd.concat(chosen_chunks, ignore_index=True)
    result.to_csv(cached, index=False)
    print(f'{name}: created {len(result):,} rows; saved {cached.name}')
    return result

analysis_datasets = {
    'UNSW': build_or_load_sample('UNSW', RANDOM_SEED),
    'CIC': build_or_load_sample('CIC', RANDOM_SEED + 1),
}

for name, frame in analysis_datasets.items():
    assert set(feature_cols + ['Label', 'Attack']).issubset(frame.columns)
    labels = pd.to_numeric(frame['Label'], errors='coerce')
    if not labels.isin([0, 1]).all():
        raise ValueError(f'{name}: Label must contain only 0 (benign) and 1 (attack)')
    print(f'{name}: records={len(frame):,}; attacks={int(labels.sum()):,}; '
          f'attack prevalence={labels.mean():.2%}')

## Create model features

We start with 41 numeric features shared by both datasets and remove four:

- **`MIN_TTL`, `MAX_TTL`:** TTL values can reflect routing paths and network settings, which may differ between environments.
- **`TCP_WIN_MAX_IN`, `TCP_WIN_MAX_OUT`:** TCP window sizes can depend on host and TCP configuration, not just the traffic activity we want to detect.

We leave these out to reduce reliance on environment-specific details. This is a design choice; it does not mean these features are always unhelpful.

We then add six features that describe the overall behavior of a network flow:

| New feature | What it describes |
|---|---|
| `FE_LOG_TOTAL_BYTES` | Total incoming and outgoing bytes, on a log scale. |
| `FE_LOG_TOTAL_PKTS` | Total incoming and outgoing packets, on a log scale. |
| `FE_BYTE_ASYMMETRY` | Whether more data travels in one direction than the other. |
| `FE_PKT_ASYMMETRY` | Whether packet counts are balanced between directions. |
| `FE_LOG_AVG_PKT_SIZE` | Approximate average bytes per packet, on a log scale. |
| `FE_LOG_DURATION` | How long the flow lasts, on a log scale. |

These features summarize traffic volume, direction, packet size, and duration. Unusual traffic may have very high packet counts, one-sided transfers, or short bursts, so these summaries may help the model notice those patterns. The log scale reduces the influence of unusually large values.

**37 retained raw features + 6 new features = 43 model features.**

In [ ]:
REMOVE_FEATURES = [
    'MIN_TTL', 'MAX_TTL', 'TCP_WIN_MAX_IN', 'TCP_WIN_MAX_OUT'
]
base_features = [f for f in feature_cols if f not in REMOVE_FEATURES]

engineered_features = [
    'FE_LOG_TOTAL_BYTES',
    'FE_LOG_TOTAL_PKTS',
    'FE_BYTE_ASYMMETRY',
    'FE_PKT_ASYMMETRY',
    'FE_LOG_AVG_PKT_SIZE',
    'FE_LOG_DURATION',
]
all_features = base_features + engineered_features

if len(base_features) != 37 or len(all_features) != 43:
    raise ValueError('Expected 37 retained raw features and 43 total features')


def engineer_features(frame):
    """Add six flow-based features."""
    result = frame.copy()
    in_bytes = result['IN_BYTES'].clip(lower=0)
    out_bytes = result['OUT_BYTES'].clip(lower=0)
    in_pkts = result['IN_PKTS'].clip(lower=0)
    out_pkts = result['OUT_PKTS'].clip(lower=0)
    total_bytes = in_bytes + out_bytes
    total_pkts = in_pkts + out_pkts

    result['FE_LOG_TOTAL_BYTES'] = np.log1p(total_bytes)
    result['FE_LOG_TOTAL_PKTS'] = np.log1p(total_pkts)
    result['FE_BYTE_ASYMMETRY'] = (
        (in_bytes - out_bytes) / (total_bytes + 1)
    )
    result['FE_PKT_ASYMMETRY'] = (
        (in_pkts - out_pkts) / (total_pkts + 1)
    )
    result['FE_LOG_AVG_PKT_SIZE'] = np.log1p(
        total_bytes / (total_pkts + 1)
    )
    result['FE_LOG_DURATION'] = np.log1p(
        result['FLOW_DURATION_MILLISECONDS'].clip(lower=0)
    )
    return result

engineered_datasets = {
    name: engineer_features(frame)
    for name, frame in analysis_datasets.items()
}

for name, frame in engineered_datasets.items():
    print(f'{name}: shape={frame.shape}; model feature count={len(all_features)}')
    print('  Missing values in engineered features:',
          int(frame[engineered_features].isna().sum().sum()))
    values = frame[engineered_features].to_numpy(dtype='float64')
    print('  Non-finite values in engineered features:',
          int((~np.isfinite(values)).sum()))
    print('  Duplicate full feature signatures:',
          int(frame.duplicated(subset=feature_cols).sum()))

## Save feature information

Save the feature names and their order in a JSON file. The dataset records stay in Google Drive.


In [ ]:
manifest = {
    'project': 'CSC786 cross-dataset NIDS',
    'source_dataset': 'NF-UNSW-NB15-v2',
    'target_dataset': 'NF-CSE-CIC-IDS2018-v2',
    'sampling': {
        'method': 'chunked Bernoulli sampling',
        'target_sample_size': TARGET_SAMPLE_SIZE,
        'source_seed': RANDOM_SEED,
        'target_seed': RANDOM_SEED + 1,
        'raw_row_counts': RAW_ROW_COUNTS,
        'cached_samples_preferred': True,
    },
    'raw_feature_count': len(feature_cols),
    'raw_features': feature_cols,
    'excluded_raw_features': REMOVE_FEATURES,
    'retained_raw_features': base_features,
    'engineered_features': engineered_features,
    'final_features_in_order': all_features,
    'label_definition': {'0': 'Benign', '1': 'Attack'},
    'notes': [
        'The target class taxonomy is not aligned with UNSW attack classes.',
        'Feature transformations are deterministic; fitted source-only scalers are introduced in later stages.',
        'No holdout records are read by this notebook.',
    ],
}
manifest_path = ANALYSIS_ROOT / 'csc786_feature_schema.json'
with manifest_path.open('w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2)

print('Feature schema saved:', manifest_path)
print('Data preparation notebook complete.')
print('No classifier training, target adaptation or holdout evaluation was performed.')